In [1]:
import os

# 取消所有代理环境变量
proxy_vars = [
    'http_proxy', 'https_proxy', 'ftp_proxy',
    'HTTP_PROXY', 'HTTPS_PROXY', 'FTP_PROXY',
    'no_proxy', 'NO_PROXY'
]

for var in proxy_vars:
    os.environ.pop(var, None)  # 使用pop避免KeyError
os.environ

environ{'LANGUAGE': 'zh_CN:zh',
        'USER': 'sukui',
        'SSH_CLIENT': '192.168.38.162 57545 22',
        'XDG_SESSION_TYPE': 'tty',
        'SHLVL': '1',
        'MOTD_SHOWN': 'pam',
        'HOME': '/home/sukui',
        'SSL_CERT_FILE': '/usr/lib/ssl/certs/ca-certificates.crt',
        'DBUS_SESSION_BUS_ADDRESS': 'unix:path=/run/user/1002/bus',
        'LOGNAME': 'sukui',
        '_': '/home/sukui/miniconda3/envs/hail_py311/bin/python',
        'XDG_SESSION_CLASS': 'user',
        'XDG_SESSION_ID': '15127',
        'VSCODE_CLI_REQUIRE_TOKEN': 'bd8ce882-c460-4a40-9eed-4572765dc05f',
        'PATH': '/home/sukui/miniconda3/envs/hail_py311/bin:/home/sukui/.vscode-server/cli/servers/Stable-7d842fb85a0275a4a8e4d7e040d2625abbf7f084/server/bin/remote-cli:/home/sukui/.local/bin:/home/sukui/bin:/home/sukui/02.software/bcftools-1.13:~/02.software/plink2:~/02.software/plink-1.07-x86_64:/home/sukui/miniconda3/envs/hail_py311/bin:/home/sukui/miniconda3/condabin:/usr/local/sbin:/usr/local

# GWAS Tutorial
This notebook is designed to provide a broad overview of Hail functionality, with emphasis on the functionality to manipulate and query a genetic dataset. we walk through a genome-wide SNP association test, and demonstrate the need to control for confounding caused by population stratification.

In [2]:
import hail as hl

hl.stop()
hl.init()

Loading BokehJS ...

25/11/12 14:17:18 WARN Utils: Your hostname, workstation-002 resolves to a loopback address: 127.0.1.1; using 192.168.41.104 instead (on interface eno1)
25/11/12 14:17:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/11/12 14:17:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Running on Apache Spark version 3.5.7
SparkUI available at http://192.168.41.104:4041
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.136-c32f88309ab0
LOGGING: writing to /home/sukui/03.projects/01.pgt/snparray_analysis/src/notebook/hail_tutorials/hail-20251112-1417-0.2.136-c32f88309ab0.log


In [3]:
# before using hail, we import some standard python lib for use throughout the notebook
from pprint import pprint

from hail.plot import show

hl.plot.output_notebook()

Loading BokehJS ...

## Download public 1000 Genomes data

We use a small chunk of the public 1000 Genomes dataset, created by downsampling the genotyped SNPs in the full VCF to about 20 Mb. We will also integrate sample and variant metadata from separate text files.

These files are hosted by the Hail team in a public Google Storage bucket; the following cell downloads that data locally.


In [4]:
hl.utils.get_1kg('data/')

2025-11-12 14:20:34.349 Hail: INFO: downloading 1KG VCF ...
  Source: https://storage.googleapis.com/hail-tutorial/1kg.vcf.bgz
2025-11-12 14:20:46.322 Hail: INFO: importing VCF and writing to matrix table...
2025-11-12 14:20:48.307 Hail: INFO: scanning VCF for sortedness...
2025-11-12 14:20:54.314 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-11-12 14:20:57.725 Hail: INFO: wrote matrix table with 10879 rows and 284 columns in 16 partitions to data/1kg.mt
2025-11-12 14:20:57.895 Hail: INFO: downloading 1KG annotations ...
  Source: https://storage.googleapis.com/hail-tutorial/1kg_annotations.txt
2025-11-12 14:20:59.571 Hail: INFO: downloading Ensembl gene annotations ...
  Source: https://storage.googleapis.com/hail-tutorial/ensembl_gene_annotations.txt
2025-11-12 14:21:08.530 Hail: INFO: Done!


## Importing data from VCF

The data in a VCF file is naturally represented as a Hail MatrixTable. By first importing the vcf file and then writing the resulting MatrixTable in Hail's native file format, all downstream operations on the VCF's data will be MUCH faster.

In [5]:
hl.import_vcf("data/1kg.vcf.bgz").write("data/1kg.mt", overwrite=True)

# Next we read the written file, assigning the variable `mt`
mt = hl.read_matrix_table('data/1kg.mt')

2025-11-12 14:24:23.585 Hail: INFO: scanning VCF for sortedness...
2025-11-12 14:24:24.728 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-11-12 14:24:28.237 Hail: INFO: wrote matrix table with 10879 rows and 284 columns in 1 partition to data/1kg.mt


## Getting to know our data

It's important to have easy ways to slice, dice, query, and summarize a dataset. Some of this functionality is demonstrated below.

The rows methods can be used to get a table with all the row fields in our MatrixTable.

We can use `rows` along with select to pull ou 5 variants. The `select` method takes either a string refering to a field name in the table, or a Hail Expression. Here, we leave the arguments blank to keep only the row key fields, `locus` and `alleles`

Use the `show` method to display the variants.

In [7]:
mt.rows().select().show(5)

,
locus,alleles
locus<GRCh37>,array<str>
1:904165,"[""G"",""A""]"
1:909917,"[""G"",""A""]"
1:986963,"[""C"",""T""]"
1:1563691,"[""T"",""G""]"
1:1707740,"[""T"",""G""]"


In [8]:
# alternatively

mt.row_key.show(5)

,
locus,alleles
locus<GRCh37>,array<str>
1:904165,"[""G"",""A""]"
1:909917,"[""G"",""A""]"
1:986963,"[""C"",""T""]"
1:1563691,"[""T"",""G""]"
1:1707740,"[""T"",""G""]"


In [9]:
# here is how to peek at the first few sample IDs
mt.s.show(5)

""
s
str
"""HG00096"""
"""HG00099"""
"""HG00105"""
"""HG00118"""
"""HG00129"""


To look at the first genotypes calls, we can use entries along with `select` and `take`. The `take` method collects the first n rows into a list. Alternatively, we can use the `show` method, which prints the first n rows to the console in a table format.

Try changing `take` to `show` in the cell below.


In [10]:
mt.entry.take(5)

[Struct(GT=Call(alleles=[0, 0], phased=False), AD=[4, 0], DP=4, GQ=12, PL=[0, 12, 147]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[8, 0], DP=8, GQ=24, PL=[0, 24, 315]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[8, 0], DP=8, GQ=23, PL=[0, 23, 230]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[7, 0], DP=7, GQ=21, PL=[0, 21, 270]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[5, 0], DP=5, GQ=15, PL=[0, 15, 205])]

## Adding column fields

A hail matrixTable can have any number of row fields and column fields for storing data associated with each row and column. Annotations are usually a critial part of any genetic study. Column fields are where you'll store information about sample phenotypes, ancestry, sex, and covariates. Row fileds can be used to store information like gene membership and functional impact for use in QC or analysis.

In this tutorial, we demonstrate how to take a text file and use it to annotate the columns in a MatrixTable

The file provided contains the sample ID, the population and "super-population" designations, the sample sex, and two simulated phenotypes (one binary, one discrete)

This file can be imported into hail with `import_table`. This function produces a Table object. Think of this as a Pandas or R dataframe that isn't limited by the memory on your machine - behind the scenes, it's distributed with Spark.

In [ ]:
table = hl.import_table('data/1kg_annotations.txt', impute=True).key_by("Sample")

# A good way to peek at the structure of a `Table` is to look at its `schema`
.describe()

2025-11-12 14:39:15.369 Hail: INFO: Reading table to impute column types


----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'Sample': str 
    'Population': str 
    'SuperPopulation': str 
    'isFemale': bool 
    'PurpleHair': bool 
    'CaffeineConsumption': int32 
----------------------------------------
Key: ['Sample']
----------------------------------------


2025-11-12 14:39:15.889 Hail: INFO: Finished type imputation
  Loading field 'Sample' as type str (imputed)
  Loading field 'Population' as type str (imputed)
  Loading field 'SuperPopulation' as type str (imputed)
  Loading field 'isFemale' as type bool (imputed)
  Loading field 'PurpleHair' as type bool (imputed)
  Loading field 'CaffeineConsumption' as type int32 (imputed)


In [13]:
# To peek at the first few values, use the `show` method:
table.show(width=100)

,,,,,
Sample,Population,SuperPopulation,isFemale,PurpleHair,CaffeineConsumption
str,str,str,bool,bool,int32
"""HG00096""","""GBR""","""EUR""",False,False,4
"""HG00097""","""GBR""","""EUR""",True,True,4
"""HG00098""","""GBR""","""EUR""",False,False,5
"""HG00099""","""GBR""","""EUR""",True,False,4
"""HG00100""","""GBR""","""EUR""",True,False,5
"""HG00101""","""GBR""","""EUR""",False,True,1
"""HG00102""","""GBR""","""EUR""",True,True,6
"""HG00103""","""GBR""","""EUR""",False,True,5


In [15]:
# Now we'll use this table to add sample annotations to our dataset, storing the annotations in col fields in our
# MatrixTable. First, we'll print the existing col schema:
print(mt.col.dtype)

# We use the `annotate_cols` method to join the table with the MT containing our dataset.
mt = mt.annotate_cols(pheno=table[mt.s])
mt.col.describe()

struct{s: str}
--------------------------------------------------------
Type:
        struct {
        s: str, 
        pheno: struct {
            Population: str, 
            SuperPopulation: str, 
            isFemale: bool, 
            PurpleHair: bool, 
            CaffeineConsumption: int32
        }
    }
--------------------------------------------------------
Source:
Index:
    ['column']
--------------------------------------------------------


In [ ]:
# 注释后的MatrixTable数据
#样本维度 (col):
# ┌─────────┬─────────────────────────────────────────────────────────┐
# │   s     │                        pheno                            │
# │         ├──────────┬───────────────┬─────────┬──────────┬─────────┤
# │         │Population│SuperPopulation│isFemale │PurpleHair│Caffeine │
# ├─────────┼──────────┼───────────────┼─────────┼──────────┼─────────┤
# │ "S001"  │  "HAN"   │     "EAS"     │  True   │  False   │    3    │
# │ "S002"  │  "YRI"   │     "AFR"     │  False  │  True    │    1    │
# │ "S003"  │  "CEU"   │     "EUR"     │  True   │  False   │    5    │
# └─────────┴──────────┴───────────────┴─────────┴──────────┴─────────┘

## Query functions and the Hail Expression Language

Hail has a number of useful query functions that can be used for gathering statistics on our dataset. These query functions take Hail Expressions as arguments.

We will start by looking at some statistics of the information in our table. The aggregate method can be used to aggregate over rows of the table.

`counter` is an aggregation function that counts the number of occurrences of each unique element. We can use this to pull out the population distribution by passing in a Hail Expression for the field that we want to count by.

In [31]:
pprint(table.aggregate(hl.agg.counter(table.SuperPopulation)))

{'AFR': 1018, 'AMR': 535, 'EAS': 617, 'EUR': 669, 'SAS': 661}


In [32]:
# `stats` is an aggregation function that produces some useful statistics about numeric collections.
# We can use this to see the distribution of the CaffeineConsumption phenotype.
pprint(table.aggregate(hl.agg.stats(table.CaffeineConsumption))) 

Struct(mean=3.9837142857142855,
       stdev=1.7021055628070711,
       min=-1.0,
       max=10.0,
       n=3500,
       sum=13943.0)


In [33]:
# However, these metrics aren't perfectly representative of the sample in our dataset.
# Her's why
table.count()

3500

In [34]:
mt.count_cols()

284

Since there are fewer samples in our dataset than in the full thousand genomes cohort, we need to look at annotations on the dataset. We can use `aggregate_cols` to get the metrics for only the samples in our dataset.

In [ ]:
mt.aggregate_cols(hl.agg.counter(mt.pheno.SuperPopulation))

2025-11-12 15:07:04.392 Hail: WARN: aggregate_cols(): Aggregates over cols ordered by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'


{'AFR': 76, 'AMR': 34, 'EAS': 72, 'EUR': 47, 'SAS': 55}

In [36]:
pprint(mt.aggregate_cols(hl.agg.stats(mt.pheno.CaffeineConsumption)))

Struct(mean=4.415492957746479,
       stdev=1.577763427465917,
       min=0.0,
       max=9.0,
       n=284,
       sum=1254.0)


The functionality demonstrated in the last few cells isn't anything especially new: it's certainly not difficult to ask these questions with Pandas or R dataframes, or even unix tools like `awk`. But Hail can use the same interfaces and query languge to analyze collections that are much larger, like the set of variants.

Here we calculate the counts of each of the 12 possible unique SNPs (4 choices for the reference base * 3 choices for the alternate base)

To do this, we need to get the alternate allele of each variant and then count the occurences of each unique ref/alt pair. this can be done with Hail's `counter` functions.

In [38]:
snp_counts = mt.aggregate_rows(hl.agg.counter(hl.Struct(ref=mt.alleles[0], alt=mt.alleles[1])))
pprint(snp_counts)

{Struct(ref='G', alt='A'): 2367,
 Struct(ref='C', alt='G'): 150,
 Struct(ref='C', alt='T'): 2418,
 Struct(ref='A', alt='T'): 75,
 Struct(ref='C', alt='A'): 494,
 Struct(ref='T', alt='C'): 1864,
 Struct(ref='T', alt='A'): 77,
 Struct(ref='G', alt='C'): 111,
 Struct(ref='G', alt='T'): 477,
 Struct(ref='A', alt='G'): 1929,
 Struct(ref='A', alt='C'): 451,
 Struct(ref='T', alt='G'): 466}


In [39]:
# We can list the counts in descending order using Python's Counter class.
from collections import Counter

counts = Counter(snp_counts)
counts.most_common()

[(Struct(ref='C', alt='T'), 2418),
 (Struct(ref='G', alt='A'), 2367),
 (Struct(ref='A', alt='G'), 1929),
 (Struct(ref='T', alt='C'), 1864),
 (Struct(ref='C', alt='A'), 494),
 (Struct(ref='G', alt='T'), 477),
 (Struct(ref='T', alt='G'), 466),
 (Struct(ref='A', alt='C'), 451),
 (Struct(ref='C', alt='G'), 150),
 (Struct(ref='G', alt='C'), 111),
 (Struct(ref='T', alt='A'), 77),
 (Struct(ref='A', alt='T'), 75)]

It's nice to see that we can actually uncover something biological from this small dataset: we see that these frequencies come in pairs. C/T and G/A are actually the same mutation, just viewed from opposite strands. Likewise, T/A and A/T are the same mutation on opposite strands. There's a 30x difference between the frequency of C/T and A/T SNPs. Why?

The same python could do this work as well, but we're starting to hit a wall - the latest genomeAD release publishes about 250 M varirants, and that won't fit in memory on a single computer.

What about genotypes? Hail can query the collection of all genotypes in the dataset, and this is getting large even for our tiny dataset. Our 284 samples and 10K variants produce 10 million unique genotypes. The gnomeA dataset has about 5 trillion unique genotypes.

Hail plotting functions allow Hail fields as arguments, so we can pass in the DP field directly here. If the range and bins arguments are not set, this function will compute the range based on minimum and maximum values of the field and use the default 50 bins.

In [40]:
p = hl.plot.histogram(
        mt.DP, range=(0,30),
        bins=30,
        title="DP Histogram",
        legend="DP"
    )
show(p)

## Quality Control

QC is where analysts spend most of their time with sequencing dataset. QC is an iterative process, and is different for every project: there is no "push-button" solution for QC. Each time the Broad collects a new group of samples, it finds new batch effects. However, by practicing open science and discussing the QC process and decisions with others, we can establish a set of best practices as a community.

QC is entirely based on the ability to understand the properties of a dataset. Hail attempts to make this easier by providing the sample_qc function, which produces a set of useful metrics and stores them in a column field.

In [41]:
mt.col.describe()

--------------------------------------------------------
Type:
        struct {
        s: str, 
        pheno: struct {
            Population: str, 
            SuperPopulation: str, 
            isFemale: bool, 
            PurpleHair: bool, 
            CaffeineConsumption: int32
        }
    }
--------------------------------------------------------
Source:
Index:
    ['column']
--------------------------------------------------------


In [42]:
mt = hl.sample_qc(mt)

In [43]:
mt.col.describe()

--------------------------------------------------------
Type:
        struct {
        s: str, 
        pheno: struct {
            Population: str, 
            SuperPopulation: str, 
            isFemale: bool, 
            PurpleHair: bool, 
            CaffeineConsumption: int32
        }, 
        sample_qc: struct {
            dp_stats: struct {
                mean: float64, 
                stdev: float64, 
                min: float64, 
                max: float64
            }, 
            gq_stats: struct {
                mean: float64, 
                stdev: float64, 
                min: float64, 
                max: float64
            }, 
            call_rate: float64, 
            n_called: int64, 
            n_not_called: int64, 
            n_filtered: int64, 
            n_hom_ref: int64, 
            n_het: int64, 
            n_hom_var: int64, 
            n_non_ref: int64, 
            n_singleton: int64, 
            n_snp: int64, 
            n_insertio

In [45]:
# - 深度和质量统计

# dp_stats: struct {    # 深度统计
#     mean: float64,    # 平均测序深度
#     stdev: float64,   # 深度标准差
#     min: float64,     # 最小深度
#     max: float64      # 最大深度
# }, 
# gq_stats: struct {    # 基因型质量统计
#     mean: float64,    # 平均基因型质量
#     stdev: float64,   # 基因型质量标准差
#     min: float64,     # 最小基因型质量
#     max: float64      # 最大基因型质量
# }

# - 呼叫率相关
# call_rate: float64,      # 呼叫率（成功分型的位点比例）
# n_called: int64,         # 成功分型的位点数
# n_not_called: int64,     # 未成功分型的位点数
# n_filtered: int64,       # 被过滤掉的位点数

# - 基因型计数
# n_hom_ref: int64,    # 纯合参考基因型数
# n_het: int64,        # 杂合基因型数  
# n_hom_var: int64,    # 纯合变异基因型数
# n_non_ref: int64,    # 非参考基因型数 (n_het + n_hom_var)

# - 变异类型统计
# n_singleton: int64,      # 单例变异数（只在单个样本中出现的变异）
# n_snp: int64,            # SNP数量
# n_insertion: int64,      # 插入变异数
# n_deletion: int64,       # 缺失变异数
# n_transition: int64,     # 转换数 (A↔G, C↔T)
# n_transversion: int64,   # 颠换数 (其他碱基替换)
# n_star: int64,           # 复杂等位基因数

# - 重要比率指标
# r_ti_tv: float64,                # 转换/颠换比率
# r_het_hom_var: float64,          # 杂合/纯合变异比率
# r_insertion_deletion: float64    # 插入/缺失比率

In [46]:
# plotting the QC metrics is a good place to start

p = hl.plot.histogram(
    mt.sample_qc.call_rate,
    range=(0.88,1),
    legend="Call Rate"
)
show(p)

In [47]:
# 基因型质量统计
p = hl.plot.histogram(mt.sample_qc.gq_stats.mean,
                      range=(10,70),
                      legend="Mean Sample GQ")
show(p)

In [48]:
# Often, these metrics are correlated
p = hl.plot.scatter(mt.sample_qc.dp_stats.mean, mt.sample_qc.call_rate,
                    xlabel="Mean DP", ylabel="Call Rate")
show(p)

In [49]:
# Removing outliers from the dataset will generally improve association results.
# We can make arbitrary cutoffs and use them to filter:
mt = mt.filter_cols(
    (mt.sample_qc.dp_stats.mean >= 4) & (mt.sample_qc.call_rate >= 0.97)
)
print(f"After filter, {mt.count_cols()} / 284 samples remain")

After filter, 250 / 284 samples remain


Next is genotype QC. It's a good idea to filter out genotypes where the reads aren't where they should be: if we find a genotype called homozygous reference with > 10% alternate reads, a genotype called homozygous alternate with > 10% reference reads, or a genotype called heterozygote without a ref / alt balance near 1:1, it is likely to be an error.

In a low-depth dataset like 1KG, it is hard to detect bad genotypes using this metric, since a read ratio of 1 alt to 10 reference can easily be explained by binomial sampling. However, in a high-depth dataset, a read ratio of 10:100 is a sure cause for concern!


In [50]:
ab = mt.AD[1] / hl.sum(mt.AD)

filter_condition_ab = (
    (mt.GT.is_hom_ref() & (ab <= 0.1))
    | (mt.GT.is_het() & (ab >= 0.25) & (ab <= 0.75))
    | (mt.GT.is_hom_var() & (ab >= 0.9))
)

fraction_filtered = mt.aggregate_entries(
    hl.agg.fraction(~filter_condition_ab)
)
print(f"Filtering {fraction_filtered*100:.2f}% entries out of downstream analysis")
mt = mt.filter_entries(filter_condition_ab)

Filtering 3.60% entries out of downstream analysis


In [51]:
# Variant QC is a bit more of the same: we can use the `variant_qc` function 
# to produce a variety of useful statistics, plot them, and filter.
mt = hl.variant_qc(mt)
mt.row.describe()

--------------------------------------------------------
Type:
        struct {
        locus: locus<GRCh37>, 
        alleles: array<str>, 
        rsid: str, 
        qual: float64, 
        filters: set<str>, 
        info: struct {
            AC: array<int32>, 
            AF: array<float64>, 
            AN: int32, 
            BaseQRankSum: float64, 
            ClippingRankSum: float64, 
            DP: int32, 
            DS: bool, 
            FS: float64, 
            HaplotypeScore: float64, 
            InbreedingCoeff: float64, 
            MLEAC: array<int32>, 
            MLEAF: array<float64>, 
            MQ: float64, 
            MQ0: int32, 
            MQRankSum: float64, 
            QD: float64, 
            ReadPosRankSum: float64, 
            set: str
        }, 
        variant_qc: struct {
            dp_stats: struct {
                mean: float64, 
                stdev: float64, 
                min: float64, 
                max: float64
            }, 

These statics actually look pretty good: we don't need to filter this dataset. Most datasets require thoughtful quality control, though. The `filter_rows` method can help!

## Let's do a GWAS

First, we need to restrict to variants that are:
- common (we'll use a cutoff of 1%)
- not so far from `Hardy-Weinberg equilibrium` as to suggest sequencing error


In [52]:
mt = mt.filter_rows(mt.variant_qc.AF[1] > 0.01)

In [53]:
mt = mt.filter_rows(mt.variant_qc.p_value_hwe > 1e-6)

In [54]:
print(f"samples:{mt.count_cols()}, Variants: {mt.count_rows()}")

samples:250, Variants: 7774


These filters removed about 15% of sites (we started with a bit over 10k). This is Not representative of most sequencing dataset! We have already downsampled the full thousand genomes dataset to include more common variants than we'd expect by chance.

In Hail, the association tests accept column fields for the sample phenotype and covariates. Since we've already got our phenotype of interest (caffeine consumption) in the dataset, we are good to go:

In [55]:
gwas = hl.linear_regression_rows(
    y=mt.pheno.CaffeineConsumption, # 表型：咖啡因摄入量
    x=mt.GT.n_alt_alleles(), # 基因型：替代等位基因计数
    covariates=[1.0] # 协变量，只有截距项
)
gwas.row.describe()

2025-11-12 16:19:34.658 Hail: INFO: linear_regression_rows: running on 250 samples for 1 response variable y,
    with input variable x, and 1 additional covariate...


--------------------------------------------------------
Type:
        struct {
        locus: locus<GRCh37>, 
        alleles: array<str>, 
        n: int32, 
        sum_x: float64, 
        y_transpose_x: float64, 
        beta: float64, 
        standard_error: float64, 
        t_stat: float64, 
        p_value: float64
    }
--------------------------------------------------------
Source:
Index:
    ['row']
--------------------------------------------------------


2025-11-12 16:19:37.157 Hail: INFO: wrote table with 7774 rows in 1 partition to /tmp/persist_TableYq5pYtfryY


Looking at the bottom of the above printout, you can see the linear regression adds new row fields for the beta, standard error, t-statistic, and p-value

Hail makes it easy to visualize results! Let's make a Manhattan plot

In [56]:
p = hl.plot.manhattan(gwas.p_value)
show(p)

This doesn't look like much of a skyline. Let's check whether our GWAS was well controlled using a Q-Q plot.

In [57]:
p = hl.plot.qq(gwas.p_value)
show(p)

2025-11-12 16:23:01.849 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-11-12 16:23:03.686 Hail: INFO: Ordering unsorted dataset with network shuffle


## Confounded

The observed p-values drift away from the expectation immediately. Either every SNP in our dataset is causally linked to caffeine consumption (unlikely), or there's a confounder.

We didn't tell you, but sample ancestry was actually used to simulate this phenotype. This leads to a stratified distribution of the phenotype. The solution is to include ancestry as covariate in our regression.

The `linear_regression_rows` function can also take column fields to use as covariates. We already annotated our samples with reported ancestry, but it is good to be skeptical of these labels due to human error. Genomes dont have that problem! instead of using reported ancestry, we will use genetic ancestry by including computed principal components in our model.

The pca function produces eigenvalues as a list and sample PCs as a table, and can also produce variant loading when asked. The `hwe_normalized_pca` function does the same, using HWE-normalized genotypes for the PCA.

In [59]:
eigenvalues, pcs, _ = hl.hwe_normalized_pca(mt.GT)

2025-11-12 16:31:44.780 Hail: INFO: hwe_normalize: found 7766 variants after filtering out monomorphic sites.
2025-11-12 16:31:47.678 Hail: INFO: pca: running PCA with 10 components...) / 1]
2025-11-12 16:31:58.440 Hail: INFO: wrote table with 0 rows in 0 partitions to /tmp/persist_TableyUekJsSlVm


In [60]:
"""
+----------+-------------------+------------------+-----+
|   s      |       PC1         |       PC2        | ... |
+----------+-------------------+------------------+-----+
| sample1  | 0.152            | -0.083           | ... |
| sample2  | -0.234           | 0.167            | ... |
| sample3  | 0.078            | 0.291            | ... |
+----------+-------------------+------------------+-----+
"""
pprint(eigenvalues)

[18.084111467840707,
 9.984076405601847,
 3.540687229805949,
 2.655598108390125,
 1.596852701724399,
 1.5405241027955296,
 1.507713504116216,
 1.4744976712480349,
 1.467690539034742,
 1.4461994473306554]


In [61]:
pcs.show(5, width=100)

,
s,scores
str,array<float64>
"""HG00096""","[1.22e-01,2.81e-01,-1.10e-01,-1.27e-01,6.68e-02,3.29e-03,-2.26e-02,4.26e-02,-9.30e-02,1.83e-01]"
"""HG00099""","[1.14e-01,2.89e-01,-1.06e-01,-6.78e-02,4.72e-02,2.87e-02,5.28e-03,-1.57e-02,1.75e-02,-1.98e-02]"
"""HG00105""","[1.09e-01,2.79e-01,-9.95e-02,-1.06e-01,8.79e-02,1.44e-02,2.80e-02,-3.38e-02,-1.08e-03,2.25e-02]"
"""HG00118""","[1.26e-01,2.95e-01,-7.58e-02,-1.08e-01,1.76e-02,7.91e-03,-5.25e-02,3.05e-02,2.00e-02,-7.78e-02]"
"""HG00129""","[1.06e-01,2.86e-01,-9.69e-02,-1.15e-01,1.03e-02,2.65e-02,-8.51e-02,2.49e-02,5.67e-02,-8.31e-03]"


Now that we've got principal components per sample, we may as well plot them! Human history exerts a strong effect in genetic datasets. Even with a 50 MB sequencing dataset, we can recover the major human populations.

In [62]:
mt = mt.annotate_cols(scores=pcs[mt.s].scores)

p = hl.plot.scatter(mt.scores[0], mt.scores[1],
                    label=mt.pheno.SuperPopulation,
                    title="PCA",
                    xlabel="PC1",
                    ylabel="PC2")
show(p)

Now we can rerun our linear regression, controlling for sample sex and the first few principal components. We'll do this with input variable the number of alternate alleles as before, and again with input variable the genotype dosage derived from the PL field.

In [63]:
gwas = hl.linear_regression_rows(
    y=mt.pheno.CaffeineConsumption,
    x=mt.GT.n_alt_alleles(),
    covariates=[1.0, mt.pheno.isFemale, mt.scores[0], mt.scores[1], mt.scores[3]]
)

2025-11-12 16:39:29.588 Hail: INFO: linear_regression_rows: running on 250 samples for 1 response variable y,
    with input variable x, and 5 additional covariates...
2025-11-12 16:39:31.192 Hail: INFO: wrote table with 7774 rows in 1 partition to /tmp/persist_TablekTlS0f6dh9


In [64]:
# we'll first make a Q-Q plot to assess inflation...
p = hl.plot.qq(gwas.p_value)
show(p)

2025-11-12 16:40:03.399 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-11-12 16:40:04.486 Hail: INFO: Ordering unsorted dataset with network shuffle


In [65]:
# That's more like it! This shape is indicative of a well-controlled (but not especially well-powered) study. 
# And now for the manhattan plot
p = hl.plot.manhattan(gwas.p_value)
show(p)

we have found a caffeine consumption locus! now simply apply Hail's Nature paper function to publish the result.

## Rare variant analysis

Here we'll demonstrate how one can use the expression language to group and count by any arbitrary properties in row and column fields. Hail also implements the sequence kerel association.

In [66]:
entries = mt.entries()
results = entries.group_by(
    pop=entries.pheno.SuperPopulation,
    chromosome=entries.locus.contig
).aggregate(
    n_het=hl.agg.count_where(entries.GT.is_het())
)

2025-11-12 16:50:30.751 Hail: WARN: entries(): Resulting entries table is sorted by '(row_key, col_key)'.
    To preserve row-major matrix table order, first unkey columns with 'key_cols_by()'


In [67]:
results.show()

2025-11-12 16:50:42.874 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
pop,chromosome,n_het
str,str,int64
"""AFR""","""1""",11039
"""AFR""","""10""",7123
"""AFR""","""11""",6777
"""AFR""","""12""",7016
"""AFR""","""13""",4650
"""AFR""","""14""",4262
"""AFR""","""15""",3847
"""AFR""","""16""",4564


We use the MatrixTable.entries method to convert our matrix table to a table (with one row for each sample for each variant). In this representation, it is easy to aggregate over any fields we like, which is often the first step of rare variant analysis.

What if we want to group by minor allele frequency bin and hair color, and calculate the mean GQ?

In [68]:
entries = entries.annotate(
    maf_bin=hl.if_else(entries.info.AF[0] < 0.01, "<1%", hl.if_else(entries.info.AF[0]<0.05, "1%-5%", ">5%"))
)

results2 = entries.group_by(
    af_bin=entries.maf_bin,
    purple_hair=entries.pheno.PurpleHair
).aggregate(
    mean_gq=hl.agg.stats(entries.GQ).mean,
    mean_dp=hl.agg.stats(entries.DP).mean
)

In [69]:
results2.show()

2025-11-12 16:58:21.901 Hail: INFO: Coerced sorted dataset          (0 + 1) / 1]


,,,
af_bin,purple_hair,mean_gq,mean_dp
str,bool,float64,float64
"""1%-5%""",False,2.48e+01,7.43e+00
"""1%-5%""",True,2.46e+01,7.47e+00
"""<1%""",False,2.35e+01,7.55e+00
"""<1%""",True,2.35e+01,7.53e+00
""">5%""",False,3.70e+01,7.65e+00
""">5%""",True,3.73e+01,7.70e+00


We've shown that it's easy to aggregate by a couple of arbitrary statistics. This specific examples may not provide especially useful pieces of information, but this same pattern can be used to detect effects of rare variation:

- Count the number of heterozygous genotypes per gene by functional category (synonymous, missense, or loss-of-function) to estimate per-gene functional constraint
- Count the number of singleton los-of-function mutation mutations per gene in cases and controls to detect genes involved in disease.

## Epilogue

For reference, here's the full workflow to all tutorial endpoints combined into one cell.

table = hl.import_table('data/1kg_annotations.txt', impute=True).key_by('Sample')

mt = hl.read_matrix_table('data/1kg.mt')
mt = mt.annotate_cols(pheno=table[mt.s])
mt = hl.sample_qc(mt)
mt = mt.filter_cols((mt.sample_qc.dp_stats.mean >= 4) & (mt.sample_qc.call_rate >= 0.97))
ab = mt.AD[1] / hl.sum(mt.AD)
filter_condition_ab = (
    (mt.GT.is_hom_ref() & (ab <= 0.1))
    | (mt.GT.is_het() & (ab >= 0.25) & (ab <= 0.75))
    | (mt.GT.is_hom_var() & (ab >= 0.9))
)
mt = mt.filter_entries(filter_condition_ab)
mt = hl.variant_qc(mt)
mt = mt.filter_rows(mt.variant_qc.AF[1] > 0.01)

eigenvalues, pcs, _ = hl.hwe_normalized_pca(mt.GT)

mt = mt.annotate_cols(scores=pcs[mt.s].scores)
gwas = hl.linear_regression_rows(
    y=mt.pheno.CaffeineConsumption,
    x=mt.GT.n_alt_alleles(),
    covariates=[1.0, mt.pheno.isFemale, mt.scores[0], mt.scores[1], mt.scores[2]],
)